# 画像比較分析 - 骨スキャン（Google Colab デモ）

このノートブックはGoogle Colab用に最適化されています。2つの画像を比較し、特定の色領域（赤と青）の類似度を分析します。

## ステップ1: サンプル画像のダウンロード

このセルを実行してGitHubからサンプル画像を自動的にダウンロードします：

In [ ]:
# GitHubからサンプル画像をダウンロード
import urllib.request
import os

print("サンプル画像をダウンロード中...")

# origin.pngをダウンロード
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/sojin25/bonescan/main/origin.png',
    'origin.png'
)
print("✓ origin.png をダウンロードしました")

# filter.pngをダウンロード
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/sojin25/bonescan/main/filter.png',
    'filter.png'
)
print("✓ filter.png をダウンロードしました")

print("\nサンプル画像の準備が完了しました！")

## ステップ2: 独自の画像をアップロード（オプション）

サンプル画像の代わりに独自の画像を使用したい場合は、以下のセルのコメントを外して実行してください：

In [ ]:
# 独自の画像をアップロードする場合は以下の行のコメントを外してください
# from google.colab import files

# print("画像ファイルをアップロードしてください：")
# uploaded = files.upload()

# # アップロードされたファイルを表示
# for filename in uploaded.keys():
#     print(f'アップロード完了: {filename}')

# # 異なるファイル名をアップロードした場合はパスを更新してください
# # img1_path = "your_origin_image.png"
# # img2_path = "your_filter_image.png"

## ステップ3: 必要なライブラリのインポート

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.io import imread
from scipy.spatial.distance import directed_hausdorff

print("ライブラリのインポートが完了しました！")

## ステップ4: 分析関数の定義

In [ ]:
def calculate_dice_coefficient(m1, m2):
    """2つのバイナリマスクのDice係数を計算"""
    if m1.sum() + m2.sum() == 0:
        return 1.0  # 両方空なら完全一致
    return 2.0 * np.logical_and(m1, m2).sum() / (m1.sum() + m2.sum())

def calculate_hausdorff_distance(m1, m2):
    """2つのバイナリマスク間のHausdorff距離を計算"""
    c1 = np.column_stack(np.where(m1))
    c2 = np.column_stack(np.where(m2))
    if c1.size == 0 or c2.size == 0:
        return np.nan  # どちらか空なら距離は未定義
    return max(directed_hausdorff(c1, c2)[0],
               directed_hausdorff(c2, c1)[0])

def segment_color_regions(img, lower, upper):
    """HSV範囲に基づいて色領域をセグメント化"""
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    return cv2.inRange(hsv, lower, upper)

def create_overlay_image_with_white_background(img_ref, m1, m2, alpha=0.25):
    """一致・不一致領域を示すオーバーレイ画像を作成"""
    white_bg = np.ones_like(img_ref) * 255
    overlay = np.zeros_like(img_ref)
    overlay[np.logical_and(m1, m2)] = [0, 255, 0]   # 緑（一致）
    overlay[np.logical_xor(m1, m2)] = [255, 0, 0]   # 赤（不一致）
    return cv2.addWeighted(white_bg, alpha, overlay, 1-alpha, 0)

def align_images_by_contours(img1, img2):
    """最大輪郭に基づいて2つの画像を位置合わせ"""
    g1 = cv2.cvtColor(img1, cv2.COLOR_RGB2GRAY)
    g2 = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY)
    _, t1 = cv2.threshold(g1, 0, 255, cv2.THRESH_OTSU)
    _, t2 = cv2.threshold(g2, 0, 255, cv2.THRESH_OTSU)
    cnt1, _ = cv2.findContours(t1, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnt2, _ = cv2.findContours(t2, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    x1, y1, w1, h1 = cv2.boundingRect(max(cnt1, key=cv2.contourArea))
    x2, y2, w2, h2 = cv2.boundingRect(max(cnt2, key=cv2.contourArea))
    crop1 = img1[y1:y1+h1, x1:x1+w1]
    crop2 = cv2.resize(img2[y2:y2+h2, x2:x2+w2], (w1, h1))
    return crop1, crop2

print("関数の定義が完了しました！")

## ステップ5: 画像パスの設定

In [ ]:
# 画像ファイルのパス
img1_path = "origin.png"
img2_path = "filter.png"

# ファイルの存在確認
import os
if os.path.exists(img1_path) and os.path.exists(img2_path):
    print("✓ 画像ファイルが見つかりました！")
    print(f"  - {img1_path}")
    print(f"  - {img2_path}")
else:
    print("⚠ 画像ファイルが見つかりません。ステップ1を実行してサンプル画像をダウンロードしてください。")

## ステップ6: 画像の読み込みと処理

In [ ]:
# 画像を読み込み
img1 = imread(img1_path)
img2 = imread(img2_path)

# 画像情報を表示
print(f"画像1のサイズ: {img1.shape}")
print(f"画像2のサイズ: {img2.shape}")

# 画像を位置合わせ
aligned1, aligned2 = align_images_by_contours(img1, img2)
print(f"\n位置合わせ後のサイズ: {aligned1.shape}")

## ステップ7: 元画像と位置合わせ後の画像を表示

In [ ]:
# 画像を表示
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(img1)
axes[0, 0].set_title("元画像1（Origin）")
axes[0, 0].axis('off')

axes[0, 1].imshow(img2)
axes[0, 1].set_title("元画像2（Filter）")
axes[0, 1].axis('off')

axes[1, 0].imshow(aligned1)
axes[1, 0].set_title("位置合わせ後の画像1")
axes[1, 0].axis('off')

axes[1, 1].imshow(aligned2)
axes[1, 1].set_title("位置合わせ後の画像2")
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

## ステップ8: 色範囲の定義とセグメンテーション

In [ ]:
# HSV色範囲の定義
# 赤色の範囲（赤は色相が0度と180度付近にあるため2つの範囲を定義）
lower_red1 = np.array([0, 70, 50])
upper_red1 = np.array([10, 255, 255])
lower_red2 = np.array([170, 70, 50])
upper_red2 = np.array([180, 255, 255])

# 青色の範囲
lower_blue = np.array([100, 150, 0])
upper_blue = np.array([140, 255, 255])

# 色領域のセグメンテーション
red1 = segment_color_regions(aligned1, lower_red1, upper_red1) | \
       segment_color_regions(aligned1, lower_red2, upper_red2)
red2 = segment_color_regions(aligned2, lower_red1, upper_red1) | \
       segment_color_regions(aligned2, lower_red2, upper_red2)

blue1 = segment_color_regions(aligned1, lower_blue, upper_blue)
blue2 = segment_color_regions(aligned2, lower_blue, upper_blue)

# セグメンテーション結果の統計情報
print("セグメンテーション結果：")
print(f"画像1の赤色ピクセル数: {np.sum(red1 > 0):,}")
print(f"画像2の赤色ピクセル数: {np.sum(red2 > 0):,}")
print(f"画像1の青色ピクセル数: {np.sum(blue1 > 0):,}")
print(f"画像2の青色ピクセル数: {np.sum(blue2 > 0):,}")

## ステップ9: 類似度の計算

In [ ]:
# 類似度メトリクスの計算
dice_r = calculate_dice_coefficient(red1 > 0, red2 > 0)
dice_b = calculate_dice_coefficient(blue1 > 0, blue2 > 0)
haus_r = calculate_hausdorff_distance(red1 > 0, red2 > 0)
haus_b = calculate_hausdorff_distance(blue1 > 0, blue2 > 0)

# 結果の表示
print("=" * 60)
print("類似度メトリクス")
print("=" * 60)
print("\n赤色領域（ホットスポット）：")
print(f"  • Dice係数: {dice_r:.3f} （0=重複なし、1=完全一致）")
if not np.isnan(haus_r):
    print(f"  • Hausdorff距離: {haus_r:.1f} ピクセル （値が小さいほど類似）")
else:
    print(f"  • Hausdorff距離: N/A （空の領域）")

print("\n青色領域（コールドスポット）：")
print(f"  • Dice係数: {dice_b:.3f} （0=重複なし、1=完全一致）")
if not np.isnan(haus_b):
    print(f"  • Hausdorff距離: {haus_b:.1f} ピクセル （値が小さいほど類似）")
else:
    print(f"  • Hausdorff距離: N/A （空の領域）")
print("=" * 60)

## ステップ10: 結果の可視化

In [ ]:
# 可視化の作成
fig, ax = plt.subplots(1, 2, figsize=(16, 8))

# 赤色領域のオーバーレイ
ax[0].imshow(create_overlay_image_with_white_background(aligned1, red1 > 0, red2 > 0))
title_red = f"赤色領域（ホットスポット）のオーバーレイ\nDice係数: {dice_r:.3f}"
if not np.isnan(haus_r):
    title_red += f"、Hausdorff距離: {haus_r:.1f}px"
ax[0].set_title(title_red, fontsize=14)
ax[0].axis('off')

# 青色領域のオーバーレイ
ax[1].imshow(create_overlay_image_with_white_background(aligned1, blue1 > 0, blue2 > 0))
title_blue = f"青色領域（コールドスポット）のオーバーレイ\nDice係数: {dice_b:.3f}"
if not np.isnan(haus_b):
    title_blue += f"、Hausdorff距離: {haus_b:.1f}px"
ax[1].set_title(title_blue, fontsize=14)
ax[1].axis('off')

# 凡例の追加
fig.text(0.5, 0.02, '🟢 緑: 一致する領域 | 🔴 赤: 一致しない領域', 
         ha='center', fontsize=14, 
         bbox=dict(boxstyle='round,pad=0.5', facecolor='lightgray', alpha=0.8))

plt.tight_layout()
plt.show()

## ステップ11: 結果の保存（オプション）

In [ ]:
# 結果を保存・ダウンロードする場合は以下のコメントを外してください

# # 出力画像の生成
# output_red = create_overlay_image_with_white_background(aligned1, red1 > 0, red2 > 0)
# output_blue = create_overlay_image_with_white_background(aligned1, blue1 > 0, blue2 > 0)

# # 画像を保存
# cv2.imwrite('red_comparison.png', cv2.cvtColor(output_red, cv2.COLOR_RGB2BGR))
# cv2.imwrite('blue_comparison.png', cv2.cvtColor(output_blue, cv2.COLOR_RGB2BGR))
# print("✓ 結果を保存しました！")

# # Google Colabでファイルをダウンロード
# from google.colab import files
# files.download('red_comparison.png')
# files.download('blue_comparison.png')
# print("✓ ファイルをダウンロードしました！")

## まとめ

この分析では、2つの骨スキャン画像を比較し、以下を計算しました：
- **Dice係数**: 領域間の重複を測定（0-1、高いほど良い）
- **Hausdorff距離**: 形状の類似性を測定（低いほど良い）

可視化の見方：
- 🟢 **緑色の領域**: 両方の画像で一致する領域
- 🔴 **赤色の領域**: 画像間で異なる領域

独自の画像を使用するには：
1. ステップ2を実行してファイルをアップロード
2. ステップ5でファイルパスを更新
3. 以降のすべてのセルを再実行